# 极客项目 ch13：激光音乐秀

依赖：pyserial/pyaudio/RPi.GPIO + 对应硬件，代码为完整忠实实现。对应《Python Playground》第 13 章（Laser Music Show）：Python 对实时音频做 FFT，三频段能量换算成双电机速度/方向经串口发给 Arduino，TB6612FNG 驱动两面旋转镜反射激光成图案；Python 代码搬运自 `code/ch13_laser.py`，Arduino 草图搬运自 `code/ch13_arduino_laser.ino`。

## 书 13.3 Arduino 程序（laser.ino）

13.3.1 配置 STBY/PWMA/AIN1/AIN2/PWMB/BIN1/BIN2 数字输出引脚；13.3.2 主循环收 5 字节协议（'H' 头 + speed1/dir1/speed2/dir2），move() 设转速与方向；13.3.3 stop() 拉低 STBY 停机。

In [ ]:
// ===== Arduino sketch：TB6612FNG 驱动双电机旋转镜面，'H' 头 5 字节串口协议（复制到 Arduino IDE 烧录） =====
// ch13_arduino_laser.ino
//
// 原书 laser.ino：经 TB6612FNG 双电机驱动板驱动两台直流电机（各带一面旋转镜反射激光）
// 引脚映射：STBY=10；电机#1（A）PWMA=3、AIN1=9、AIN2=8；电机#2（B）PWMB=5、BIN1=11、BIN2=12
// 串口协议：'H' 头校验 + 4 字节（speed1, dir1, speed2, dir2），9600 波特

int STBY = 10; //standby

// Motor #1
int PWMA = 3; //Speed control
int AIN1 = 9; //Direction
int AIN2 = 8; //Direction

// Motor #2
int PWMB = 5; //Speed control
int BIN1 = 11; //Direction
int BIN2 = 12; //Direction

void setup(){

  pinMode(STBY, OUTPUT);

  pinMode(PWMA, OUTPUT);
  pinMode(AIN1, OUTPUT);
  pinMode(AIN2, OUTPUT);

  pinMode(PWMB, OUTPUT);
  pinMode(BIN1, OUTPUT);
  pinMode(BIN2, OUTPUT);

  // initialize the serial communication:
  Serial.begin(9600);
}

// main loop that reads motor data sent by laser.py
void loop()
{
  // The data sent is of the form:
  // 'H' (header), speed1, dir1, speed2, dir2
  if (Serial.available() >= 5) {
    if (Serial.read() == 'H') {
      // read next 4 bytes
      byte s1 = Serial.read();
      byte d1 = Serial.read();
      byte s2 = Serial.read();
      byte d2 = Serial.read();

      // stop motor if both speeds are 0
      if (s1 == 0 && s2 == 0) {
        stop();
      }
      else {
        // set speed and direction of both motors
        move(0, s1, d1);
        move(1, s2, d2);
      }
      // small pause
      delay(20);
    }
    else {
      // invalid data, stop motors
      stop();
    }
  }
  else {
    // no data
    delay(250);
  }
}

// set motor speed and direction
// motor: #1 -> 1, #2 -> 0
// direction: 1/0
void move(int motor, int speed, int direction)
{
  // disable standby
  digitalWrite(STBY, HIGH);

  boolean inPin1 = LOW;
  boolean inPin2 = HIGH;

  if (direction == 1) {
    inPin1 = HIGH;
    inPin2 = LOW;
  }

  if (motor == 1) {
    digitalWrite(AIN1, inPin1);
    digitalWrite(AIN2, inPin2);
    analogWrite(PWMA, speed);
  }
  else {
    digitalWrite(BIN1, inPin1);
    digitalWrite(BIN2, inPin2);
    analogWrite(PWMB, speed);
  }
}

// bring motors to a stop
void stop(){
  // enable standby
  digitalWrite(STBY, LOW);
}

## 书 13.4 Python 代码

读音频（pyaudio 输入流）→ numpy rfft → 三个频段平均振幅 → 换算 speed1/speed2/dir1/dir2 → struct.pack('BBBBB') 发串口；先 --mtest/--atest 测电机，再进 FFT 主循环（Ctrl-C 退出时关音频流、停电机）。运行：`python ch13_laser.py --port COM3 [--mtest | --atest]`。

In [ ]:
# ===== 书 13.2 节：所需模块（Python 侧依赖） =====
import sys
import serial
import struct
import pyaudio
import numpy
import math
from time import sleep
import argparse

In [ ]:
# ===== 书 13.4.6/13.4.8 节：电机测试（autoTest 自动扫描 / manualTest 手动输入） =====
# manual test of motor direction and speeds
def manualTest(ser):
    print('starting manual test...')
    try:
        while True:
            print('enter motor control info such as < 100 1 120 0 >')
            strIn = input()  # 原书为 raw_input()，Python 3 改 input()
            vals = [int(val) for val in strIn.split()[:4]]
            vals.insert(0, ord('H'))  # 头部插入 'H' 凑足 5 字节协议
            data = struct.pack('BBBBB', *vals)
            ser.write(data)
    except:
        print('exiting...')
        # shut off the motors
        vals = [ord('H'), 0, 1, 0, 1]
        data = struct.pack('BBBBB', *vals)
        ser.write(data)
        ser.close()


# automatic test for sending motor speeds
def autoTest(ser):
    print('starting automatic test...')
    try:
        while True:
            # for each direction combination
            for dr in [(0, 0), (1, 0), (0, 1), (1, 1)]:
                # for a range of speeds
                for j in range(25, 180, 10):
                    for i in range(25, 180, 10):
                        vals = [ord('H'), i, dr[0], j, dr[1]]
                        print(vals[1:])
                        data = struct.pack('BBBBB', *vals)
                        ser.write(data)
                        sleep(0.1)
    except KeyboardInterrupt:
        print('exiting...')
        # shut off motors
        vals = [ord('H'), 0, 1, 0, 1]
        data = struct.pack('BBBBB', *vals)
        ser.write(data)
        ser.close()

In [ ]:
# ===== 书 13.4.1 节：选择 pyaudio 输入设备 =====
# get pyaudio input device
def getInputDevice(p):
    index = None
    nDevices = p.get_device_count()
    print('Found %d devices. Select input device:' % nDevices)
    for i in range(nDevices):
        deviceInfo = p.get_device_info_by_index(i)
        devName = deviceInfo['name']
        print("%d: %s" % (i, devName))
    try:
        index = int(input())
    except:
        pass
    if index is not None:
        devName = p.get_device_info_by_index(index)["name"]
        print("Input device chosen: %s" % devName)
    return index

In [ ]:
# ===== 书 13.4.2–13.4.5 节：打开音频流 → FFT → 频段振幅 → 电机指令 =====
# fft of live audio
def fftLive(ser):
    p = pyaudio.PyAudio()
    inputIndex = getInputDevice(p)

    fftLen = 2**11      # 2048 个样本（2 的幂，FFT 对此优化）
    sampleRate = 44100

    print('opening stream...')
    stream = p.open(format=pyaudio.paInt16,
                    channels=1,
                    rate=sampleRate,
                    input=True,
                    frames_per_buffer=fftLen,
                    input_device_index=inputIndex)
    try:
        while True:
            # read a chunk of data
            data = stream.read(fftLen)
            # convert to numpy array
            dataArray = numpy.frombuffer(data, dtype=numpy.int16)

            # get FFT of data
            fftVals = numpy.fft.rfft(dataArray)*2.0/fftLen
            # get absolute values of complex numbers
            fftVals = numpy.abs(fftVals)
            # get average of 3 frequency bands
            # 0-100 Hz, 100-1000 Hz and 1000-2500 Hz
            # （bin 宽 = 44100/2048 ≈ 21.5 Hz：这里实际按 rfft 下标粗分三段，与注释的 Hz 数并不一一对应）
            levels = [numpy.sum(fftVals[0:100])/100,
                      numpy.sum(fftVals[100:1000])/900,
                      numpy.sum(fftVals[1000:2500])/1500]

            # The data sent is of the form:
            # 'H' (header), speed1, dir1, speed2, dir2
            vals = [ord('H'), 100, 1, 100, 1]

            # The code below sets speed/direction based on information
            # in the frequency bin. This is totally arbitrary - depends
            # on the effect you are trying to create. The goal is to
            # generate 4 values (s1 [0-255], d1 [0/1], s2 [0-255], d2 [0/1])
            # from the constantly changing frequency content, so the motors
            # are in sync with the music.

            # speed1
            vals[1] = int(5*levels[0]) % 255
            # speed2
            vals[3] = int(100 + levels[1]) % 255

            # dir
            d1 = 0
            if levels[2] > 0.1:
                d1 = 1
            vals[2] = d1
            vals[4] = 0

            # pack data
            data = struct.pack('BBBBB', *vals)
            # write data to serial port
            ser.write(data)
            # a slight pause
            sleep(0.001)
    except KeyboardInterrupt:
        print('stopping...')
    finally:
        print('cleaning up')
        stream.close()
        p.terminate()
        # shut off motors
        vals = [ord('H'), 0, 1, 0, 1]
        data = struct.pack('BBBBB', *vals)
        ser.write(data)
        # close serial
        ser.flush()
        ser.close()

In [ ]:
# ===== 书 13.4.7 节：命令行选项与 main() 分发 =====
# main method
def main():
    parser = argparse.ArgumentParser(description='Analyzes audio input and '
                                     'sends motor control information via serial port')
    parser.add_argument('--port', dest='serial_port_name', required=True)
    parser.add_argument('--mtest', action='store_true', default=False)
    parser.add_argument('--atest', action='store_true', default=False)
    args = parser.parse_args()

    strPort = args.serial_port_name
    print('opening ', strPort)
    ser = serial.Serial(strPort, 9600)
    if args.mtest:
        manualTest(ser)
    elif args.atest:
        autoTest(ser)
    else:
        fftLive(ser)


if __name__ == '__main__':
    main()